# Chapter 6: Going sparse — Mixture-of-Experts (+ 2024 refinements)

Chapter 4 took you from the 2019 GPT-2 design to the **2023 Llama** design (RoPE, RMSNorm, SwiGLU, GQA). This chapter takes the next step to the **2024–25 frontier**. The single biggest architectural shift in that window — the thing DeepSeek-V3, Llama-4, Qwen3, Mixtral, and GPT-OSS all share — is going **sparse** with a **Mixture-of-Experts (MoE)**.

### The one idea: decouple *parameters* from *compute*

A dense model uses **every** parameter for **every** token. That ties knowledge capacity (params) to cost (FLOPs) — to know more, you must compute more.

MoE breaks that link. Replace the one MLP in each block with **N expert MLPs** plus a tiny **router**. For each token, the router picks the **top-k** experts (e.g. 2 of 8) and only *those* run. So:

```
8 experts in the layer  →  8x the FFN parameters (capacity)
top-2 routing           →  each token still pays for only 2 (≈ constant compute)
```

You get a much bigger model that costs about the same **per token**. That's why frontier labs scaled to hundreds of billions of MoE params while keeping inference affordable.

### What's new vs Chapter 4

| # | Component | Ch.4 (dense Llama) | Ch.6 (sparse, 2024-25) | The benefit |
|---|---|---|---|---|
| 1 | **FFN** | one SwiGLU MLP | **MoE**: N SwiGLU experts + a router, top-k per token | more capacity at ~constant compute/token |
| 2 | **Routing health** | n/a | **load-balancing auxiliary loss** | stops the router collapsing onto a few experts (the thing that makes MoE *train*) |
| 3 | **Attention stability** | q·k raw | **QK-Norm** (RMSNorm on q,k) | tames attention-logit blow-ups; now standard in 2024 models |
| + | **Attention KV** (concept) | GQA | **MLA** (DeepSeek latent attention) | compresses the KV cache below even GQA — covered as concept |

### The honest caveat (read first)

At **~50M-class / 8 GB**, a small MoE **will not beat the dense model on quality** — MoE's advantage shows up at *scale*, where capacity is the bottleneck. So this chapter teaches the **mechanism** and **measures** it: params go up, **active params / FLOPs per token stay flat**, and an **expert-utilization plot** shows the load-balancing loss doing its job. That's the real lesson — the *how* and *why* of the frontier's dominant trick, not a quality win at toy scale.

**Pattern unchanged:** concept → `TODO` → `# check`. The two genuinely new pieces — **top-k gating** and the **load-balance loss** — are the TODOs; the dense Llama parts you built in Ch.4 are given.

## 0. Setup + the MoE `Config`

New fields:
- **`n_experts`** — how many expert MLPs per layer (the capacity multiplier).
- **`top_k`** — how many experts each token uses (the compute knob). `top_k < n_experts` is what makes it sparse.
- **`moe_d_ff`** — hidden width of *each* expert (smaller than the dense `d_ff`, since there are now many).
- **`use_qk_norm`** — toggle QK-Norm.
- **`aux_coef`** — weight on the load-balancing loss (small, e.g. 0.01).

In [ ]:
import math, os, time
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
import tiktoken

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
DATA_DIR = "data"
enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token
VOCAB_SIZE = enc.n_vocab


@dataclass
class Config:
    vocab_size: int = VOCAB_SIZE
    d_model: int = 512
    n_heads: int = 8
    n_kv_heads: int = 2
    n_layers: int = 8
    block_size: int = 256
    rope_theta: float = 10000.0
    dropout: float = 0.0
    # --- MoE ---
    n_experts: int = 8         # experts per layer (capacity)
    top_k: int = 2             # experts used per token (compute)
    moe_d_ff: int = 768        # hidden width PER expert (small -- there are n_experts of them)
    aux_coef: float = 0.01     # load-balancing loss weight
    use_qk_norm: bool = True
    def __post_init__(self):
        assert self.d_model % self.n_heads == 0
        assert self.n_heads % self.n_kv_heads == 0
        assert self.top_k <= self.n_experts

cfg = Config()
print(cfg)
print(f"each token uses {cfg.top_k}/{cfg.n_experts} experts -> ~{cfg.top_k/cfg.n_experts:.0%} of FFN params per token")

## Given: the dense Llama pieces from Chapter 4

Reproduced so the notebook stands alone — RoPE, RMSNorm, GQA building blocks, and **`SwiGLU` (now used as a single *expert*)**. Nothing new here; you built and understood all of it in Ch.4.

In [ ]:
def precompute_rope(d_k, max_pos, theta=10000.0, device="cpu"):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_k, 2, device=device) / d_k))
    t = torch.arange(max_pos, device=device)
    freqs = torch.outer(t, inv_freq)
    emb = torch.cat([freqs, freqs], dim=-1)
    return emb.cos(), emb.sin()

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(q, k, cos, sin):
    return q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin

def repeat_kv(x, n_rep):
    if n_rep == 1:
        return x
    b, n_kv, t, d = x.shape
    return x[:, :, None, :, :].expand(b, n_kv, n_rep, t, d).reshape(b, n_kv * n_rep, t, d)

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x / rms) * self.weight

class SwiGLU(nn.Module):
    """One expert = one SwiGLU MLP (Ch.4). bias-free."""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
        self.down.RESIDUAL_SCALE_INIT = True
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

## 1. The router + top-k gating

The **router** is a single `Linear(d_model -> n_experts)`: for each token it produces one score per expert. We then keep the **top-k** experts and turn their scores into weights with a softmax *over the chosen k* (not all N) — so a token's two experts get weights that sum to 1.

```
router_logits[token] = [2.1, -0.4, 3.3, 0.1, ...]   # one score per expert
top-2 -> experts {2, 0}                              # the indices
softmax over their scores -> weights [0.77, 0.23]    # how much each contributes
```

The token's output is then `0.77 * expert2(x) + 0.23 * expert0(x)`. Only those two experts ever run for this token — that's the sparsity.

| variable | shape | meaning |
|---|---|---|
| `router_logits` | `(N, n_experts)` | per-token score for every expert (N = tokens in the batch) |
| `gate_weights` | `(N, top_k)` | softmax weights over the chosen experts (sum to 1 per token) |
| `expert_idx` | `(N, top_k)` | which experts each token picked |

In [ ]:
def top_k_gating(router_logits, top_k):
    """Pick the top_k experts per token and weight them.

    Returns (gate_weights, expert_idx), each (N, top_k).
    Steps:
      1. topk_val, expert_idx = router_logits.topk(top_k, dim=-1)     # (N, top_k)
      2. gate_weights = softmax(topk_val, dim=-1)                     # normalize OVER THE CHOSEN k
      3. return gate_weights, expert_idx
    """
    # TODO:
    raise NotImplementedError

In [ ]:
# check -- shapes, weights sum to 1 per token, indices valid
torch.manual_seed(0)
_logits = torch.randn(100, cfg.n_experts)
_gw, _idx = top_k_gating(_logits, cfg.top_k)
assert _gw.shape == (100, cfg.top_k) and _idx.shape == (100, cfg.top_k), "wrong gating shapes"
assert torch.allclose(_gw.sum(-1), torch.ones(100), atol=1e-5), "gate weights must sum to 1 per token"
assert _idx.max() < cfg.n_experts and _idx.min() >= 0, "expert indices out of range"
# the top weight must correspond to the largest logit
assert (_logits.argmax(-1) == _idx[:, 0]).float().mean() > 0.99, "expert_idx[:,0] should be the argmax expert"
print("ok: top-k gating picks the right experts and weights them to sum 1")

## 2. The load-balancing auxiliary loss — the thing that makes MoE train

Left alone, routing **collapses**: a few experts get picked early, get better, get picked more, and the rest die (the "rich get richer" failure). The fix (Switch Transformer) is an **auxiliary loss** that pushes routing toward *uniform* expert usage. Add it to the main loss with a small weight.

For each expert `i`, take two quantities over the batch:
- `f_i` = the **fraction of token-slot assignments** that went to expert `i` (how often it was actually picked),
- `P_i` = the **mean router probability** mass on expert `i` (how much the router *wanted* it, on average).

```
aux = n_experts * sum_i ( f_i * P_i )
```

Why it works: this product is minimized (= 1.0) exactly when **both** are uniform (`f_i = P_i = 1/n_experts`), and grows as routing concentrates. Multiplying the *hard* assignment count `f` by the *soft* probability `P` keeps it differentiable (gradients flow through `P` to the router). Minimizing it spreads tokens evenly across experts.

| variable | shape | meaning |
|---|---|---|
| `router_logits` | `(N, n_experts)` | the raw router scores |
| `expert_idx` | `(N, top_k)` | which experts were chosen (from gating) |
| `aux` | scalar | the load-balance penalty (≈ 1.0 when balanced, up to ~n_experts when collapsed) |

In [ ]:
def load_balance_loss(router_logits, expert_idx, n_experts):
    """Switch-Transformer load-balancing aux loss. Returns a scalar.

    Steps:
      1. probs = softmax(router_logits, dim=-1)                  # (N, n_experts)
      2. P = probs.mean(dim=0)                                   # (n_experts,) mean prob per expert
      3. onehot = F.one_hot(expert_idx, n_experts).float()       # (N, top_k, n_experts)
         f = onehot.sum(dim=(0, 1)) / (N * top_k)                # (n_experts,) fraction of assignments
      4. return n_experts * (f * P).sum()
    """
    # TODO:
    raise NotImplementedError

In [ ]:
# check -- uniform routing -> aux ~ 1.0 (the minimum); collapsed routing -> aux much larger
N, E, K = 2000, cfg.n_experts, cfg.top_k
# (a) balanced: random logits route ~uniformly
torch.manual_seed(0)
_lg = torch.randn(N, E)
_, _idx = top_k_gating(_lg, K)
_aux_bal = load_balance_loss(_lg, _idx, E)
# (b) collapsed: logits that always favor expert 0
_lg_bad = torch.zeros(N, E); _lg_bad[:, 0] = 10.0
_, _idx_bad = top_k_gating(_lg_bad, K)
_aux_bad = load_balance_loss(_lg_bad, _idx_bad, E)
print(f"aux (balanced) = {_aux_bal:.3f}   aux (collapsed) = {_aux_bad:.3f}")
assert _aux_bal < 1.6, "balanced routing should give aux near the ~1.0 minimum"
assert _aux_bad > _aux_bal + 1.0, "collapsed routing must be penalized much more than balanced"
print("ok: the aux loss is low when balanced and high when collapsed")

## 3. The MoE layer (given — it uses your two TODOs)

Now assemble: `n_experts` SwiGLU experts + the router, with `top_k_gating` choosing experts and `load_balance_loss` keeping them balanced. The dispatch loop is mechanical — for each expert, gather the tokens routed to it, run it once on that slice, and scatter the weighted result back. We also keep a running `tok_per_expert` count so we can plot utilization later.

> **ponytail:** this per-expert Python loop is the *simple, correct* dispatch — it runs each expert on only its tokens. Production MoE uses fused grouped-GEMM kernels for speed, but the semantics are exactly this. Don't reach for the fancy kernel to learn the idea.

In [ ]:
class MoE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_experts = cfg.n_experts
        self.top_k = cfg.top_k
        self.router = nn.Linear(cfg.d_model, cfg.n_experts, bias=False)
        self.experts = nn.ModuleList([SwiGLU(cfg.d_model, cfg.moe_d_ff) for _ in range(cfg.n_experts)])
        self.register_buffer("tok_per_expert", torch.zeros(cfg.n_experts), persistent=False)

    def forward(self, x):
        B, T, d = x.shape
        xf = x.reshape(-1, d)                                   # (N, d) flatten tokens
        router_logits = self.router(xf)                        # (N, n_experts)
        gate_w, idx = top_k_gating(router_logits, self.top_k)  # (N, k), (N, k)

        out = torch.zeros_like(xf)
        for e in range(self.n_experts):                        # run each expert on ONLY its tokens
            sel = (idx == e)                                   # (N, k) where expert e was chosen
            if sel.any():
                tok, slot = sel.nonzero(as_tuple=True)         # token rows routed to e
                w = gate_w[tok, slot].unsqueeze(-1)            # their gate weights
                out.index_add_(0, tok, w * self.experts[e](xf[tok]))

        with torch.no_grad():                                  # track utilization for the plot
            self.tok_per_expert += torch.bincount(idx.flatten(), minlength=self.n_experts).float()
        aux = load_balance_loss(router_logits, idx, self.n_experts)
        return out.reshape(B, T, d), aux

In [ ]:
# check -- MoE preserves shape, returns a scalar aux, and is genuinely SPARSE (total >> active params)
_moe = MoE(cfg)
_x = torch.randn(2, 6, cfg.d_model)
_out, _aux = _moe(_x)
assert _out.shape == _x.shape and _aux.dim() == 0, "MoE must return (same-shape out, scalar aux)"
_total = sum(p.numel() for p in _moe.experts.parameters())
_active = _total * cfg.top_k / cfg.n_experts                  # only top_k of n_experts run per token
print(f"expert params: total {_total/1e6:.1f}M  |  active/token ~{_active/1e6:.1f}M  ({cfg.top_k}/{cfg.n_experts})")
assert _active < _total * 0.6, "with top_k < n_experts, active params must be well below total -- that's the point"
print("ok: MoE is sparse -- big capacity, small per-token compute")

## 4. QK-Norm — a small 2024 stability upgrade

As models get deeper/wider, the raw dot products `q·k` going into attention can grow large, pushing the softmax into saturation and destabilizing training. **QK-Norm** is the now-standard fix (Chameleon, Gemma-2-era, many others): apply an **RMSNorm over each head's `q` and `k`** (the `d_k` dimension) *before* RoPE and attention. It keeps the query/key magnitudes controlled so attention logits stay in a sane range — cheap insurance, especially with bf16.

In [ ]:
def apply_qk_norm(q, k, q_norm, k_norm):
    """RMSNorm each head's q and k over the head dim (last dim). q,k: (B, n_heads/n_kv, T, d_k).
    q_norm, k_norm are RMSNorm(d_k) modules. Return (q_normed, k_normed)."""
    # TODO: return q_norm(q), k_norm(k)
    raise NotImplementedError

In [ ]:
# check -- after QK-norm each head vector has RMS ~1 along the head dim
_qn, _kn = RMSNorm(64), RMSNorm(64)
_q = torch.randn(2, 8, 5, 64) * 4.0
_qg, _kg = apply_qk_norm(_q, _q.clone(), _qn, _kn)
_rms = _qg.pow(2).mean(-1).sqrt()
assert torch.allclose(_rms, torch.ones_like(_rms), atol=1e-2), "QK-norm should make per-head RMS ~1"
print("ok: QK-norm controls q/k magnitude per head")

## 5. Assemble the MoE model (given)

A `MoEBlock` is the Ch.4 block with two swaps: attention gains **QK-Norm**, and the dense MLP becomes the **MoE** layer (which also emits an aux loss). The model collects the aux loss from every layer and returns it alongside the usual logits/loss; the training loop adds `aux_coef * aux` to the cross-entropy. Everything else — RoPE, residual pattern, weight tying, KV-cache generation — is unchanged from Ch.4.

In [ ]:
class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg.n_heads; self.n_kv = cfg.n_kv_heads
        self.n_rep = cfg.n_heads // cfg.n_kv_heads; self.d_k = cfg.d_model // cfg.n_heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_heads * self.d_k, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.d_k, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.d_k, bias=False)
        self.o_proj = nn.Linear(cfg.n_heads * self.d_k, cfg.d_model, bias=False)
        self.o_proj.RESIDUAL_SCALE_INIT = True
        self.use_qk_norm = cfg.use_qk_norm
        if cfg.use_qk_norm:
            self.q_norm = RMSNorm(self.d_k); self.k_norm = RMSNorm(self.d_k)

    def forward(self, x, cos, sin, past_kv=None, use_cache=False, is_causal=True):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv, self.d_k).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv, self.d_k).transpose(1, 2)
        if self.use_qk_norm:
            q, k = apply_qk_norm(q, k, self.q_norm, self.k_norm)    # <-- the 2024 tweak, before RoPE
        q, k = apply_rope(q, k, cos, sin)
        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2); v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None
        k = repeat_kv(k, self.n_rep); v = repeat_kv(v, self.n_rep)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal,
            dropout_p=cfg.dropout if self.training else 0.0)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1)), present


class MoEBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model); self.attn = Attention(cfg)
        self.mlp_norm = RMSNorm(cfg.d_model); self.mlp = MoE(cfg)
    def forward(self, x, cos, sin, past_kv=None, use_cache=False, is_causal=True):
        a, present = self.attn(self.attn_norm(x), cos, sin, past_kv, use_cache, is_causal)
        x = x + a
        m, aux = self.mlp(self.mlp_norm(x))
        return x + m, aux, present


class MoEGPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg; self.block_size = cfg.block_size; self.n_layers = cfg.n_layers
        self.wte = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([MoEBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight
        d_k = cfg.d_model // cfg.n_heads
        cos, sin = precompute_rope(d_k, cfg.block_size, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            std = 0.02
            if getattr(m, "RESIDUAL_SCALE_INIT", False):
                std /= math.sqrt(2 * self.n_layers)
            nn.init.normal_(m.weight, mean=0.0, std=std)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        x = self.drop(self.wte(idx))
        aux_total = 0.0
        for block in self.blocks:
            x, aux, _ = block(x, cos, sin, is_causal=True)
            aux_total = aux_total + aux
        logits = self.lm_head(self.norm_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss, aux_total / self.n_layers


@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=0.7, top_k=50):
    model.eval()
    caches = [None] * len(model.blocks)
    def step(tokens, start, is_causal):
        T = tokens.shape[1]
        cos, sin = model.rope_cos[start:start + T], model.rope_sin[start:start + T]
        x = model.wte(tokens)
        for i, b in enumerate(model.blocks):
            x, _, caches[i] = b(x, cos, sin, past_kv=caches[i], use_cache=True, is_causal=is_causal)
        return model.lm_head(model.norm_f(x))[:, -1, :]
    pos = idx.shape[1]; logits = step(idx, 0, True)
    for _ in range(max_new_tokens):
        logits = logits / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, top_k); logits[logits < v[:, [-1]]] = float("-inf")
        nxt = torch.multinomial(F.softmax(logits, -1), 1); idx = torch.cat([idx, nxt], 1)
        if nxt.item() == EOT: break
        logits = step(nxt, pos, False); pos += 1
    model.train(); return idx

def complete(model, prompt, n=40, temperature=0.7, top_k=50):
    ids = torch.tensor([enc.encode_ordinary(prompt)], device=device)
    return enc.decode(generate(model, ids, n, temperature, top_k)[0].tolist())

In [ ]:
# check -- model builds, reports total vs active params, first loss ~ ln(V), aux is present
_m = MoEGPT(cfg).to(device)
_total = sum(p.numel() for p in _m.parameters())
_expert = sum(p.numel() for b in _m.blocks for p in b.mlp.experts.parameters())
_active = _total - _expert + _expert * cfg.top_k / cfg.n_experts    # non-expert + only top_k experts
print(f"TOTAL params: {_total/1e6:.1f}M   |   ACTIVE per token: ~{_active/1e6:.1f}M   "
      f"(experts: {_expert/1e6:.1f}M total, {cfg.top_k}/{cfg.n_experts} active)")
_idx = torch.randint(0, VOCAB_SIZE, (2, cfg.block_size), device=device)
_tgt = torch.randint(0, VOCAB_SIZE, (2, cfg.block_size), device=device)
with torch.autocast(device_type=device, dtype=torch.bfloat16):
    _lg, _loss, _aux = _m(_idx, _tgt)
print(f"first loss {_loss.item():.3f} (ln V={math.log(VOCAB_SIZE):.2f}) | aux {_aux.item():.3f}")
assert 9.5 < _loss.item() < 12.0 and _aux.item() > 0, "fresh loss ~ln(V) and a positive aux expected"
print("ok: sparse MoE model builds -- total params up, active-per-token down")
del _m

## 6. Train it — and watch the experts balance

Standard pretraining loop (given, like 3a) with one addition: the total loss is **`cross_entropy + aux_coef * aux`**. We reuse whatever token cache you already built in a previous chapter (no new download) and train a short run — enough to see the loss fall and the **expert-utilization** even out.

Watch two things: the **loss** (should fall normally) and, afterward, the **expert-utilization histogram** — flat-ish bars mean the load-balancing loss is working; a few towering bars would mean routing collapsed (raise `aux_coef`).

In [ ]:
# reuse an existing token cache from a previous chapter (no new download)
_candidates = ["mix_edu_100M.bin", "mix_web_120M.bin", "fineweb_train_500M.bin", "fineweb_train.bin", "fineweb_tiny.bin"]
TRAIN_BIN = next((os.path.join(DATA_DIR, c) for c in _candidates if os.path.exists(os.path.join(DATA_DIR, c))), None)
assert TRAIN_BIN, "no token cache found -- run Ch.3a or Ch.4 first to build one in data/"
data = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
print(f"training on {TRAIN_BIN} ({len(data):,} tokens)")

def get_batch(data, block, bs):
    ix = torch.randint(len(data) - block - 1, (bs,))
    x = torch.stack([torch.from_numpy(data[i:i+block].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block].astype(np.int64)) for i in ix])
    if device == "cuda":
        return x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    return x.to(device), y.to(device)

def get_lr(step, max_lr, min_lr, warmup, max_steps):
    if step < warmup: return max_lr * step / warmup
    if step >= max_steps: return min_lr
    frac = (step - warmup) / (max_steps - warmup)
    return min_lr + 0.5 * (1 + math.cos(math.pi * frac)) * (max_lr - min_lr)

In [ ]:
from tqdm.auto import tqdm

torch.manual_seed(1337)
model = MoEGPT(cfg).to(device)
for b in model.blocks: b.mlp.tok_per_expert.zero_()      # reset utilization counters
print(f"params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

USE_COMPILE = False   # the per-expert dispatch loop is data-dependent -> compile helps little here; keep it off
if USE_COMPILE and device == "cuda":
    try:
        import triton  # noqa
        model = torch.compile(model)
    except Exception:
        pass

MAX_STEPS, BATCH, BLOCK = 4000, 16, cfg.block_size
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1)
hist = {"step": [], "loss": [], "aux": []}
pbar = tqdm(range(MAX_STEPS), desc="moe", dynamic_ncols=True)
for step in pbar:
    for g in opt.param_groups: g["lr"] = get_lr(step, 3e-4, 3e-5, int(0.03*MAX_STEPS), MAX_STEPS)
    x, y = get_batch(data, BLOCK, BATCH)
    opt.zero_grad(set_to_none=True)
    with torch.autocast(device_type=device, dtype=torch.bfloat16):
        _, loss, aux = model(x, y)
        total = loss + cfg.aux_coef * aux                # <-- CE + load-balancing penalty
    total.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    hist["step"].append(step); hist["loss"].append(loss.item()); hist["aux"].append(aux.item())
    pbar.set_postfix(loss=f"{loss.item():.3f}", aux=f"{aux.item():.3f}", lr=f"{opt.param_groups[0]['lr']:.1e}")
    if step % 500 == 0:
        tqdm.write(f"step {step:5d} | loss {loss.item():.3f} | aux {aux.item():.3f}")
pbar.close()
torch.save({"model": getattr(model, '_orig_mod', model).state_dict(), "cfg": cfg}, os.path.join(DATA_DIR, "moe.pt"))
print("saved data/moe.pt")

In [ ]:
# dashboard: loss, aux loss, and the EXPERT-UTILIZATION histogram (did load-balancing work?)
import matplotlib.pyplot as plt
base = getattr(model, "_orig_mod", model)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

s = hist["step"]
ma = np.convolve(hist["loss"], np.ones(50)/50, mode="valid")
ax[0].plot(s, hist["loss"], alpha=0.3); ax[0].plot(s[49:], ma, color="tab:blue")
ax[0].axhline(math.log(VOCAB_SIZE), ls="--", color="gray"); ax[0].set_title("loss"); ax[0].set_xlabel("step")

ax[1].plot(s, hist["aux"], color="tab:orange", alpha=0.6)
ax[1].axhline(1.0, ls="--", color="gray", label="balanced=1.0")
ax[1].set_title("load-balance aux loss"); ax[1].set_xlabel("step"); ax[1].legend()

# total tokens routed to each expert, summed over training and all layers
counts = torch.stack([b.mlp.tok_per_expert for b in base.blocks]).sum(0).cpu().numpy()
ax[2].bar(range(cfg.n_experts), counts / counts.sum())
ax[2].axhline(1/cfg.n_experts, ls="--", color="gray", label="perfectly uniform")
ax[2].set_title("expert utilization (flat = balanced)"); ax[2].set_xlabel("expert"); ax[2].set_ylabel("token share"); ax[2].legend()
plt.tight_layout(); plt.show()
print("share per expert:", np.round(counts/counts.sum(), 3))

## 7. The frontier in concept — MLA and friends (no code)

MoE + QK-Norm are the two upgrades we *built*. A few more define the 2024–25 frontier, worth knowing even though we won't implement them here:

- **MLA — Multi-head Latent Attention (DeepSeek-V2/V3).** GQA shrank the KV cache by using fewer KV heads. MLA goes further: it **compresses K and V into a small shared low-rank "latent" vector** that's cached instead of full K/V, then projects them back up inside attention. The cache becomes *smaller than GQA's* while keeping near-MHA quality — the current state of the art in attention efficiency, paired with MoE in DeepSeek-V3.
- **Fine-grained + shared experts (DeepSeek-MoE).** Many *small* experts (more specialization) plus one or two **always-on "shared" experts** that capture common patterns so the routed experts can specialize. A refinement of the plain MoE you built.
- **Sliding-window attention (Mistral).** Each token attends only to the last `W` tokens, making long-context attention linear in length; combined with a few global layers.
- **Logit soft-capping (Gemma-2).** Squash attention and final logits with `cap * tanh(x/cap)` to prevent blow-ups — another stability trick alongside QK-Norm.

All of them are the *same move* you've been making since Ch.4: find a place the dense/uniform design wastes compute, memory, or stability — and make it sparse, compressed, or normalized.

## Done — you built the frontier's defining trick

| change | what it does | the catch at our scale |
|---|---|---|
| **MoE** | N experts, top-k per token → big capacity, ~constant compute/token | only pays off when capacity is the bottleneck (i.e. at scale) |
| **load-balance loss** | keeps all experts in use | needs tuning (`aux_coef`); too low → collapse, too high → hurts the LM loss |
| **QK-Norm** | controls attention-logit magnitude | basically free; helps most at depth/precision pressure |

**The takeaway:** you can now read a frontier model card — "37B active / 671B total params, MLA, fine-grained MoE" — and know exactly what every term means and why it's there. The dense→sparse jump is the single most important architectural idea separating 2023 from 2025 models.

### Where the project stands
Architecture is now genuinely modern. The remaining handoff end goal is **multilingual** (a tokenizer + per-language slices through Ch.4's mixture loader). Everything else — bigger base, more tokens, DPO, RAG — is scale and refinement on the stack you've built across six chapters.